# CRISP-DM Phase 3 — Data Preparation (3-class Apple-Watch pipeline)

Turns the raw Kaggle Apple-Watch recordings into model-ready data using the **current
`src/ml4b/` modules** — the *same* code the training script and the app use. This
notebook only **imports and orchestrates**; no pipeline logic is duplicated.

Pipeline: load (lag trimmed, canonical units) -> sliding window (200 @ 100 Hz, 50%
overlap) -> activity gate (rest) -> invariant features -> augmentation, with
**leave-one-set-out** grouping for honest evaluation.

ADRs: 006 (window), 016 (classes), 017 (gate), 018 (features), 019 (augmentation),
021 (evaluation).

> Requires `data/raw/kaggle_gym_imu/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ml4b.data.kaggle_loader import load_kaggle_3class, TARGET_CLASSES
from ml4b.data.canonical import WINDOW_SIZE, OVERLAP, TARGET_HZ
from ml4b.data.windowing import apply_sliding_window
from ml4b.data.activity_gate import gate_window_df, ACCEL_MAG_STD_THRESHOLD, GYRO_MAG_MEAN_THRESHOLD
from ml4b.data.features_invariant import extract_invariant_features, feature_columns
from ml4b.data.augmentation import augment_windows

print('window size:', WINDOW_SIZE, '| overlap:', OVERLAP, '| target Hz:', TARGET_HZ)
print('target classes:', TARGET_CLASSES)

## 1. Load the 3-class data
`load_kaggle_3class()` reads only the files mapping to our three classes, trims the
~0.24 s CoreMotion warm-up lag, reconstructs total acceleration in g + gyro in rad/s,
and tags each file with a `recording_id` (one set = one group).

In [ ]:
raw_df = load_kaggle_3class()
print('rows:', f'{len(raw_df):,}', '| columns:', list(raw_df.columns))
print('sets:', raw_df['recording_id'].nunique())
print('\nsets per class:')
print(raw_df.groupby('exercise_name')['recording_id'].nunique().to_string())

## 2. Sliding window (200 samples = 2 s @ 100 Hz, 50% overlap)
Windows never cross a set boundary, and `recording_id` is carried through so we can
group by set during evaluation (ADR-006).

In [ ]:
window_df = apply_sliding_window(raw_df, window_size=WINDOW_SIZE, overlap=OVERLAP, sampling_rate=TARGET_HZ)
print('windows:', f'{len(window_df):,}')
print('columns:', list(window_df.columns))
print('\nwindows per class:')
print(window_df['exercise_name'].value_counts().to_string())

## 3. Activity gate (rest is NOT a trained class)
A window is *active* if `accel-magnitude std` OR `gyro-magnitude mean` clears the
thresholds; otherwise it is `rest`. Rest is detected by signal energy, not learned, so
it transfers across devices/users (ADR-017). On the (all-exercise) Kaggle windows the
vast majority are correctly kept active.

In [ ]:
active = gate_window_df(window_df)
print(f'accel-std threshold: {ACCEL_MAG_STD_THRESHOLD} g | gyro-mean threshold: {GYRO_MAG_MEAN_THRESHOLD} rad/s')
print(f'active (kept) windows: {active.mean()*100:.1f}%  |  gated as rest: {(~active).mean()*100:.1f}%')

## 4. Invariant features (39 device-robust features)
Magnitude statistics + spectral, per-window z-normalized shape features, and axis-pair
correlations — robust to watch orientation and per-device scale (ADR-018).

In [ ]:
feats = extract_invariant_features(window_df)
feature_names = feature_columns(feats)
print('feature count:', len(feature_names))
print('feature names:', feature_names)
feats[['exercise_name', 'recording_id'] + feature_names[:4]].head()

## 5. Augmentation (subject-diversity substitute)
The anchor is single-subject, so training synthesises variability with composed
**rotation + time-warp + mirror + jitter** (5 copies -> 6x; ADR-019). Demonstrated here
on a small subset to keep the notebook fast — `scripts/train_model.py` augments the
full training folds.

In [ ]:
demo = window_df.head(20)
aug = augment_windows(demo, n_augment=5, random_state=42)
print(f'demo windows: {len(demo)} -> after 5x augmentation: {len(aug)} (originals kept first)')
# Rotation is norm-preserving: |accel| of an augmented copy matches an original window's stats only loosely
# (time-warp/jitter also applied), but the label and recording_id are preserved:
print('recording_id preserved on augmented rows:', aug['recording_id'].notna().all())

## 6. Leave-one-set-out grouping (honest, leakage-free)
Because the data is single-subject, we cannot do leave-one-*subject*-out. Instead each
**set** (`recording_id`) is held out once; augmented copies of a held-out set are
excluded from its training fold (ADR-021). This is set up here and used in
**04_modeling**.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
groups = feats['recording_id'].to_numpy()
logo = LeaveOneGroupOut()
n_splits = logo.get_n_splits(groups=groups)
print('number of leave-one-set-out folds (= number of sets):', n_splits)
# Show the first fold's train/test sizes as a sanity check (no set spans both).
X = feats[feature_names].to_numpy(); y = feats['exercise_name'].to_numpy()
tr, te = next(logo.split(X, y, groups))
print(f'fold 0 -> train windows: {len(tr)}, test windows: {len(te)} '
      f'(held-out set: {feats.iloc[te]["recording_id"].iloc[0]})')

## 7. Summary
- Raw -> windowed (200 @ 100 Hz) -> gated -> 39 invariant features, grouped by set.
- Augmentation expands training data 6x to stand in for missing subject diversity.
- Evaluation uses leave-one-set-out (75 folds).

Next: **04_modeling** trains the Random Forest and evaluates it with this grouping.